In [1]:
import polars as pl

rna_seq_sources = ['Hugo et al.', 'Kwong et al.', 'Yan et al.']
q_pcr_sources = ['Louveau et al.']
micro_array_sources = ['Long et al.', 'Rizos et al.']

source_map = {
    'doi:10.1016/j.cell.2015.07.061': 'Hugo et al.',
    'doi:10.1172/JCI78954DS1': 'Kwong et al.',
    'doi:10.1158/1078-0432.CCR-18-0720': 'Yan et al.',
    'doi:10.3390/cancers11081203': 'Louveau et al.',
    'doi:10.1038/ncomms6694': 'Long et al.',
    'doi:10.1158/1078-0432.CCR-13-3122': 'Rizos et al.'
}

gex = pl.read_csv("../dataset/original/gene_expressions.csv")
gex = gex.with_columns(pl.col('source').replace(source_map))

In [2]:
DROP_NULLS = True
SELECT_PRE_TREATMENT = True
SELECT_RNA_SEQ = False

DATASET = 'gex_pre'

In [3]:
if SELECT_PRE_TREATMENT == True:
    gex = gex.filter((pl.col('temporality') == 'pre treatment'))
if SELECT_RNA_SEQ == True:
    gex = gex.filter(pl.col('source').is_in(rna_seq_sources))

In [4]:
# Drop useless features
gex = gex.drop(['id', 'creation_datetime', 'GeneID', 'description', 'temporality'])
gex.select(pl.all().null_count())

patientID,sample_id,HGNC,value,source
u32,u32,u32,u32,u32
92181,0,207000,0,0


In [5]:
if DROP_NULLS == True:
    # Sample is useless if patientID or HGNC is not given
    gex = gex.drop_nulls(subset=['patientID', 'HGNC'])
    gex.select(pl.all().null_count())

In [6]:
gex = gex.with_columns(
    pl.when(pl.col('source').is_in(rna_seq_sources))
    .then(pl.lit('RNA-seq'))
    .when(pl.col('source').is_in(micro_array_sources))
    .then(pl.lit('micro-array'))
    .otherwise(pl.lit('qPCR'))
    .alias('Method')
)
gex

patientID,sample_id,HGNC,value,source,Method
str,str,str,f64,str,str
"""LM_1""","""LMSAM_1""","""BRAF""",7.60798,"""Louveau et al.""","""qPCR"""
"""LM_1""","""LMSAM_1""","""RAF1""",16.095204,"""Louveau et al.""","""qPCR"""
"""LM_1""","""LMSAM_1""","""ARAF""",4.1515,"""Louveau et al.""","""qPCR"""
"""LM_1""","""LMSAM_1""","""PDGFRB""",1.199885,"""Louveau et al.""","""qPCR"""
"""LM_1""","""LMSAM_1""","""IGF1R""",5.47246,"""Louveau et al.""","""qPCR"""
…,…,…,…,…,…
"""HL_Shi-43""","""Pt17-baseline""","""ZYG11A""",0.0625681,"""Hugo et al.""","""RNA-seq"""
"""HL_Shi-43""","""Pt17-baseline""","""ZYG11B""",5.74608,"""Hugo et al.""","""RNA-seq"""
"""HL_Shi-43""","""Pt17-baseline""","""ZYX""",45.907933,"""Hugo et al.""","""RNA-seq"""


In [7]:
gex.write_csv(f'../dataset/created/gex/{DATASET}.csv')